# Dataset prediction and research evidence
This notebook demonstrates the new workflow using saved models and completed evaluations. It does not retrain the production models.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'peptide_lab').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from peptide_lab.config import RESULTS
from peptide_lab.datasets import catalog,browse,attach_labels
from peptide_lab.pipeline import ScreeningPipeline

## 1. Raw-data preprocessing audit
Canonical amino-acid filtering, exact deduplication and explicit length rules are reproducible from the raw file. The original empty notebook remains preserved.

In [2]:
print(json.dumps(json.loads((RESULTS/'preprocessing_audit.json').read_text()),indent=2))

{
  "raw_records": 51345,
  "missing_sequences": 0,
  "invalid_sequence_records": 0,
  "duplicate_valid_records_removed": 44585,
  "canonical_unique_sequences": 6760,
  "length_counts": {
    "5_40": 6154,
    "5_60": 6510
  },
  "matches_historical_clean_set": true,
  "raw_sha256": "9e33122afb3bdd169e8bed620b1040bf1601d60cf25b6c8c8d9870bf37522b6f",
  "clean_sha256": "a7e9516a39b3f9521cac27dd3d8f33b85ef9c0dde2ba1d18cc7003f6b7dfc24e",
  "policy": "Strip and uppercase sequence; retain 20 canonical residues; deduplicate by sequence preserving first appearance; apply explicit length bounds. No species or modification filter. Sequence-only models do not model chemical modifications or assay conditions."
}


## 2. Browse labeled dataset peptides
Use a toxicity test subset. A missing activity label must remain missing; toxicity and activity are independent tasks.

In [3]:
data=catalog()
selected=browse(data,'Toxicity benchmark','test',min_length=5,max_length=40).sample(5,random_state=42)
display(selected)

,sequence,in_grampa,activity_label,activity_split,toxicity_label,toxicity_split,length
425,WWDGECRLWSNGCRKHKECCSNHCKGIYCDIW,False,<NA>,not in benchmark,1,test,32
75,DDECEPPGVFCGTFKIGPPCCSGWCFLWCA,False,<NA>,not in benchmark,1,test,30
181,GVGCSSGCHKVGGQCRCG,False,<NA>,not in benchmark,1,test,18
30,ASRACNPPCTGLSMCQAGRCGYIRF,False,<NA>,not in benchmark,1,test,25
364,SCNNSCQQHSQCASHCVCLLNKCRTVN,False,<NA>,not in benchmark,1,test,27


## 3. Predict activity and toxicity, then compare available labels
Join labels by sequence rather than row order because predictions are ranked.

In [4]:
pipe=ScreeningPipeline()
result=attach_labels(pipe.predict(selected.sequence.tolist()),selected)
display(result[['sequence','activity_score','toxicity_score','toxicity_label','toxicity_matches_label','toxicity_split']])

,sequence,activity_score,toxicity_score,toxicity_label,toxicity_matches_label,toxicity_split
0,SCNNSCQQHSQCASHCVCLLNKCRTVN,0.992138,0.377037,1,False,test
1,GVGCSSGCHKVGGQCRCG,0.999989,0.834209,1,True,test
2,ASRACNPPCTGLSMCQAGRCGYIRF,0.993666,0.976071,1,True,test
3,WWDGECRLWSNGCRKHKECCSNHCKGIYCDIW,0.999733,0.998533,1,True,test
4,DDECEPPGVFCGTFKIGPPCCSGWCFLWCA,0.999233,0.998912,1,True,test


## 4. Does the VAE use its latent vector?
Compare encoder means with zeroed/shuffled representations under the same true prefixes. This tests model dependence, not biological meaning.

In [5]:
research=json.loads((RESULTS/'extended/research_summary.json').read_text())
display(pd.DataFrame(research['latent_diagnostics']))

,latent_condition,cross_entropy,token_accuracy,teacher_forced_exact_match,free_running_exact_match
0,encoder_mean,1.728813,0.473808,0.010753,NaN
1,zero_latent,3.594186,0.139882,0.000000,NaN
2,shuffled_latent,4.419851,0.109674,0.000000,NaN
3,free_running_from_encoder_mean,NaN,NaN,NaN,0.010753


## 5. Predictor bias, uncertainty and ranking sensitivity
Inspect class counts and balanced accuracy. Eight short negative activity examples cannot support broad generalization claims.

In [6]:
display(pd.DataFrame(research['predictor_checks']))
display(pd.DataFrame(research['confidence_intervals']))
display(pd.DataFrame(research['ranking_sensitivity']))

,task,evaluation,n,positive,negative,accuracy,balanced_accuracy,roc_auc,brier_score
0,activity,selected_all,8877,2563,6314,0.961924,0.949707,0.991557,0.028662
1,activity,length_only_all,8877,2563,6314,0.923285,0.899830,0.967234,0.058125
2,activity,selected_length_5_40,1817,1809,8,0.996698,0.625000,0.985351,0.002931
3,toxicity,selected_all,580,269,311,0.922414,0.920373,0.971802,0.063970
4,toxicity,length_only_all,580,269,311,0.581034,0.584223,0.603970,0.243446
5,toxicity,selected_length_5_40,433,225,208,0.914550,0.915780,0.966645,0.070731


,task,metric,lower_95,upper_95,bootstrap_replicates
0,activity,accuracy,0.958094,0.965926,300
1,activity,roc_auc,0.990025,0.993058,300
2,toxicity,accuracy,0.898276,0.941379,300
3,toxicity,roc_auc,0.959474,0.982804,300


,activity_weight,toxicity_weight,top_10_overlap_with_product,spearman_rank_correlation
0,0.25,0.75,0.7,0.995349
1,0.50,0.50,1.0,0.999994
2,0.75,0.25,0.9,0.991315


## 6. Completed controlled toy study
Ten seeds assess subset/initialization robustness; three matched seeds compare latent dimensions and word dropout. These are new experiments with weak small-sample uncertainty, not historical results or a full-data hyperparameter guarantee.

In [7]:
runs=pd.read_csv(RESULTS/'extended/toy_study_runs.csv')
print('Completed runs:',len(runs))
display(runs)
summary=pd.read_csv(RESULTS/'extended/toy_study_summary.csv')
display(summary[summary.metric=='test_cross_entropy'])

,seed,latent,word_dropout,test_cross_entropy,test_kl,token_accuracy,teacher_forced_exact_match,uniqueness,exact_novelty_vs_5_40_pool,natural_end_rate,mean_length,mean_entropy
0,42,24,0.2,2.481675,2.262588,0.228947,0.0,1.0,1.0,0.92,21.35,2.839103
1,101,24,0.2,2.418887,2.448594,0.275449,0.0,1.0,1.0,0.96,20.52,2.723113
2,123,24,0.2,2.369682,2.273680,0.267442,0.0,1.0,1.0,0.96,16.53,2.634958
3,256,24,0.2,2.711506,1.003506,0.143266,0.0,1.0,1.0,1.00,18.55,2.884126
4,512,24,0.2,2.595186,2.019145,0.235294,0.0,1.0,1.0,0.99,18.50,2.654057
5,777,24,0.2,2.584351,1.623635,0.222535,0.0,1.0,1.0,0.95,20.02,2.762358
6,999,24,0.2,2.540666,2.573038,0.249389,0.0,1.0,1.0,0.99,18.49,2.784393
7,2024,24,0.2,2.678986,1.627206,0.210900,0.0,1.0,1.0,0.99,17.15,2.690576
8,3141,24,0.2,2.368802,2.441480,0.240000,0.0,1.0,1.0,0.98,18.30,2.691458
9,4096,24,0.2,2.494338,1.892372,0.232911,0.0,1.0,1.0,0.97,16.45,2.598814


,latent,word_dropout,seed_scope,runs,metric,mean,std,bootstrap_mean_lower95,bootstrap_mean_upper95
0,16,0.2,all_available_seeds,3,test_cross_entropy,2.428383,0.054531,2.365440,2.461351
7,24,0.0,all_available_seeds,3,test_cross_entropy,2.430496,0.052376,2.387043,2.488653
14,24,0.2,all_available_seeds,10,test_cross_entropy,2.524408,0.120086,2.456910,2.594687
21,24,0.2,matched_first_three_seeds,3,test_cross_entropy,2.423415,0.056134,2.369682,2.481675
28,32,0.2,all_available_seeds,3,test_cross_entropy,2.440955,0.039203,2.401887,2.480292
35,64,0.2,all_available_seeds,3,test_cross_entropy,2.436698,0.030722,2.412919,2.471386


Completed runs: 22


## 7. Demonstration
Run `python -m streamlit run app.py`. Choose Dataset explorer → Toxicity benchmark → test → Predict dataset activity & toxicity. Then inspect a peptide, export results and open Prediction history. The Research checks page explains the completed studies.

Reproduce additional evidence with `EVALUATE_PROJECT.bat`. Missing original historical notebooks and wet-lab validation cannot be reconstructed from model files.